# EpiPINN SIR-R0 with NVIDIA PhysicsNeMo and CUDA

This notebook refactors the original SciANN Case 1 workflow into a PhysicsNeMo/PyTorch PINN implementation. It keeps the original synthetic SIR data generation, joint training logic, plots, error metrics, and saved outputs while using PhysicsNeMo `FullyConnected`, `PDE`, and `PhysicsInformer` to organize the model and physics residuals.

中文说明：本 Notebook 保留原始 `EpiPINN-SIR-R0.ipynb` 的 Case 1 常数传播率 SIR 逻辑，用 NVIDIA PhysicsNeMo 组织神经网络、PDE 约束、损失和训练过程，并自动检测 CUDA；有 GPU 时用 CUDA，没有 GPU 时回退 CPU。

In [ ]:
import os
import pickle
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.integrate import odeint
from sympy import Function, Number, Symbol

import physicsnemo
from physicsnemo.models.mlp.fully_connected import FullyConnected
from physicsnemo.sym.eq.pde import PDE
from physicsnemo.sym.eq.phy_informer import PhysicsInformer

# Keep Warp/PhysicsNeMo cache out of the project tree and unwritable home cache paths.
DEFAULT_CACHE_DIR = Path(os.environ.get("TMPDIR", "/tmp")) / "physicsnemo-warp-cache"
os.environ.setdefault("WARP_CACHE_PATH", str(DEFAULT_CACHE_DIR))
os.environ.setdefault("XDG_CACHE_HOME", str(Path(os.environ.get("TMPDIR", "/tmp")) / "physicsnemo-cache"))

SEED = 1234

def set_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 70)
print("Environment check")
print("=" * 70)
print("Python executable :", os.sys.executable)
print("PhysicsNeMo       :", getattr(physicsnemo, "__version__", "unknown"))
print("PyTorch           :", torch.__version__)
print("PyTorch CUDA      :", torch.version.cuda)
print("CUDA available    :", torch.cuda.is_available())
print("Selected device   :", device)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU name          :", props.name)
    print("GPU memory        :", f"{props.total_memory / 1024**3:.2f} GiB")
else:
    print("CUDA is not available. The notebook will run on CPU.")

## Case 1: Constant Transmission Rate

The epidemiological setup and scaling below match the original notebook:

- `N = 56e6`, `delta = 1/5`, `r0 = 3`, so the true `beta = delta * r0 = 0.6`.
- `C = 1e5` scales the population variables.
- `C1 = tf * C / N` and `C2 = tf * delta` are the scaled ODE coefficients.

In [ ]:
# SIR model parameters, matching the original SciANN notebook.
N = 56e6
delta = 1 / 5
r0 = 3.0
beta = delta * r0

t0 = 0.0
tf = 90.0

C = 1e5
C1 = tf * C / N
C2 = tf * delta

print(f"True beta: {beta:.6f}")
print(f"C1: {C1:.6f}, C2: {C2:.6f}")

In [ ]:
def SIR(x, t, delta, beta, N, t0):
    """Reference unscaled SIR ODE used only to generate synthetic data."""
    del t0
    S, I, R = x
    lambda_val = beta * I / N
    dSdt = -lambda_val * S
    dIdt = lambda_val * S - delta * I
    dRdt = delta * I
    return [dSdt, dIdt, dRdt]

# Initial conditions.
S0 = N - 1
I0 = 1
R0 = 0
x0 = [S0, I0, R0]

# Time span and reference solution.
timespan = np.arange("2020-02-01", "2020-05-01", dtype="datetime64[D]")
tspan = timespan.astype(int)
x = odeint(SIR, x0, tspan, args=(delta, beta, N, tspan[0]))

S_data = x[:, 0]
I_data = x[:, 1]
R_data = x[:, 2]

# Poisson observations of infectious individuals, matching the original workflow.
I_obs = np.random.poisson(I_data)

plt.figure(figsize=(10, 10))
plt.subplot(2, 1, 1)
plt.plot(timespan, S_data, "b", label="Susceptible")
plt.plot(timespan, R_data, "g", label="Recovered")
plt.legend()
plt.xlabel("date")
plt.ylabel("individuals")
plt.title("Susceptible and Recovered")

plt.subplot(2, 1, 2)
plt.plot(timespan, I_data, "r", label="Infectious")
plt.plot(timespan, I_obs, "xm", label="samples")
plt.legend()
plt.xlabel("date")
plt.ylabel("individuals")
plt.title("Infectious and Observed Data")
plt.tight_layout()
plt.show()

In [ ]:
# Scale data.
t_data = np.arange(t0, tf)
t_test = np.arange(t0, tf, 0.1)

I_obs_sc = I_obs / C
I_data_sc = I_data / C
t_data_sc = t_data / tf
t_test_sc = t_test / tf

weekly = False
if weekly:
    I_obs_train_sc = I_obs_sc[::7]
    t_data_train_sc = t_data_sc[::7]
else:
    I_obs_train_sc = I_obs_sc
    t_data_train_sc = t_data_sc

print("Number of data points:", len(t_data_train_sc))

## PhysicsNeMo PINN Components

The original SciANN implementation used two functionals `Ss(ts)` and `Is(ts)`, a non-negative constant `Beta`, and the algebraic conservation relation `Rs = N/C - Ss - Is`. The refactor keeps that model logic:

- `FullyConnected` from PhysicsNeMo defines the neural networks.
- `torch.square` reproduces the original square output activation for non-negative `Ss` and `Is`.
- `Beta` is optimized as a scalar raw parameter and mapped through `softplus` for non-negativity.
- `SIRScaledPDE` and `PhysicsInformer` compute the ODE residuals with automatic differentiation.

In [ ]:
class SIRScaledPDE(PDE):
    """Scaled SIR residuals for PhysicsNeMo PhysicsInformer.

    Variable naming follows PhysicsNeMo's derivative convention. The scaled time
    coordinate is named x so the residuals use Ss__x, Is__x, and Rs__x.
    """

    def __init__(self, C1, C2):
        self.dim = 1
        x = Symbol("x")
        Ss = Function("Ss")(x)
        Is = Function("Is")(x)
        Rs = Function("Rs")(x)
        Beta = Function("Beta")(x)
        self.equations = {
            "L_dSdt": Ss.diff(x) + Number(float(C1)) * Beta * Is * Ss,
            "L_dIdt": Is.diff(x) - Number(float(C1)) * Beta * Is * Ss + Number(float(C2)) * Is,
            "L_dRdt": Rs.diff(x) - Number(float(C2)) * Is,
        }


class SIRPhysicsNeMoPINN(torch.nn.Module):
    """Joint SIR PINN with PhysicsNeMo MLPs for Ss and Is."""

    def __init__(self, hidden_size=50, num_layers=4, beta_init=0.2):
        super().__init__()
        self.s_net = FullyConnected(
            in_features=1,
            out_features=1,
            layer_size=hidden_size,
            num_layers=num_layers,
            activation_fn="tanh",
        )
        self.i_net = FullyConnected(
            in_features=1,
            out_features=1,
            layer_size=hidden_size,
            num_layers=num_layers,
            activation_fn="tanh",
        )
        # inverse softplus so softplus(raw_beta) starts near beta_init.
        beta_init_tensor = torch.tensor(float(beta_init))
        raw_init = torch.log(torch.expm1(beta_init_tensor))
        self.raw_beta = torch.nn.Parameter(raw_init.clone().detach())

    def beta(self):
        return torch.nn.functional.softplus(self.raw_beta)

    def forward(self, ts):
        Ss = torch.square(self.s_net(ts))
        Is = torch.square(self.i_net(ts))
        Rs = N / C - Ss - Is
        Beta = self.beta().expand_as(Ss)
        return {"Ss": Ss, "Is": Is, "Rs": Rs, "Beta": Beta}


pde = SIRScaledPDE(C1=C1, C2=C2)
physics_informer = PhysicsInformer(
    required_outputs=["L_dSdt", "L_dIdt", "L_dRdt"],
    equations=pde,
    grad_method="autodiff",
    device=device,
)

print("PhysicsInformer required inputs:", sorted(physics_informer.required_inputs))
pde.pprint()

In [ ]:
def column(values, *, dtype=np.float32):
    return np.asarray(values, dtype=dtype).reshape(-1, 1)


def make_tensor(values, requires_grad=False):
    return torch.tensor(column(values), dtype=torch.float32, device=device, requires_grad=requires_grad)


def log_collocation(t0, tf, count):
    points = np.random.uniform(np.log1p(t0 / tf), np.log1p(1.0), count)
    return np.exp(points) - 1.0


def relative_l2(reference, prediction):
    reference = np.asarray(reference).reshape(-1)
    prediction = np.asarray(prediction).reshape(-1)
    return np.linalg.norm(reference - prediction, 2) / np.linalg.norm(reference, 2)


def mse_zero(tensor):
    return torch.mean(torch.square(tensor))


def compute_losses(model, physics_informer, t_data_batch, i_obs_batch, t_ode_batch):
    # Data loss, equivalent to sn.Data(Is) on the observed infection samples.
    pred_data = model(t_data_batch)
    loss_data = torch.mean(torch.square(pred_data["Is"] - i_obs_batch))

    # ODE residual loss, equivalent to the three SciANN PDE residuals.
    t_ode_batch = t_ode_batch.detach().clone().requires_grad_(True)
    pred_ode = model(t_ode_batch)
    residuals = physics_informer.forward({
        "coordinates": t_ode_batch,
        "Ss": pred_ode["Ss"],
        "Is": pred_ode["Is"],
        "Rs": pred_ode["Rs"],
        "Beta": pred_ode["Beta"],
    })
    loss_ode_s = mse_zero(residuals["L_dSdt"])
    loss_ode_i = mse_zero(residuals["L_dIdt"])
    loss_ode_r = mse_zero(residuals["L_dRdt"])
    loss_ode = loss_ode_s + loss_ode_i + loss_ode_r

    # Initial-condition losses. This keeps the original S0/I0/R0 constraints.
    t_initial = torch.zeros((1, 1), dtype=torch.float32, device=device, requires_grad=True)
    pred_initial = model(t_initial)
    target_s0 = torch.tensor([[S0 / C]], dtype=torch.float32, device=device)
    target_i0 = torch.tensor([[I0 / C]], dtype=torch.float32, device=device)
    target_r0 = torch.tensor([[R0 / C]], dtype=torch.float32, device=device)
    loss_ic_s = torch.mean(torch.square(pred_initial["Ss"] - target_s0))
    loss_ic_i = torch.mean(torch.square(pred_initial["Is"] - target_i0))
    loss_ic_r = torch.mean(torch.square(pred_initial["Rs"] - target_r0))
    loss_ic = loss_ic_s + loss_ic_i + loss_ic_r

    return {
        "data": loss_data,
        "ode": loss_ode,
        "ode_s": loss_ode_s,
        "ode_i": loss_ode_i,
        "ode_r": loss_ode_r,
        "ic": loss_ic,
        "ic_s": loss_ic_s,
        "ic_i": loss_ic_i,
        "ic_r": loss_ic_r,
    }


def sample_batch(t_data_tensor, i_obs_tensor, t_ode_tensor, batch_size):
    data_count = t_data_tensor.shape[0]
    ode_count = t_ode_tensor.shape[0]
    data_idx = torch.randint(0, data_count, (min(batch_size, data_count),), device=device)
    ode_idx = torch.randint(0, ode_count, (batch_size,), device=device)
    return t_data_tensor[data_idx], i_obs_tensor[data_idx], t_ode_tensor[ode_idx]

## Joint Training

The default values match the original notebook (`Nc=6000`, `epochs_joint=5000`, `batch_size=100`). For a quick CUDA smoke test, temporarily set `epochs_joint` to a small value such as `20` before running the cell.

In [ ]:
# Training parameters.
loss_err = "mse"
optimizer_name = "adam"
learning_rate = 1e-3
adaptive_NTK = {"method": "gradient_norm", "freq": 100}

Nc = 6000
epochs_joint = 5000
batch_size = 100

# Training points: observed data times plus log-biased ODE collocation points.
t_train_ode = log_collocation(t0, tf, Nc)

t_data_tensor = make_tensor(t_data_train_sc)
i_obs_tensor = make_tensor(I_obs_train_sc)
t_ode_tensor = make_tensor(t_train_ode)

model = SIRPhysicsNeMoPINN(hidden_size=50, num_layers=4, beta_init=0.2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Loss weights. The gradient-norm update is a lightweight PyTorch analogue of
# SciANN's adaptive multi-objective weighting, kept explicit for readability.
loss_weights = {"data": 1.0, "ode": 1.0, "ic": 1.0}
history = {
    "loss": [],
    "loss_data": [],
    "loss_ode": [],
    "loss_ic": [],
    "Beta": [],
    "w_data": [],
    "w_ode": [],
    "w_ic": [],
}

print("Model device:", next(model.parameters()).device)
print("Training data device:", t_data_tensor.device)
print("ODE collocation device:", t_ode_tensor.device)

In [ ]:
def grad_norm_for_loss(loss, parameters):
    grads = torch.autograd.grad(loss, parameters, retain_graph=True, allow_unused=True)
    norms = [g.detach().norm() for g in grads if g is not None]
    if not norms:
        return torch.tensor(0.0, device=device)
    return torch.stack(norms).mean()


def update_loss_weights(losses, model, momentum=0.9):
    params = [p for p in model.parameters() if p.requires_grad]
    norms = {
        "data": grad_norm_for_loss(losses["data"], params),
        "ode": grad_norm_for_loss(losses["ode"], params),
        "ic": grad_norm_for_loss(losses["ic"], params),
    }
    norm_values = torch.stack([torch.clamp(v, min=1e-12) for v in norms.values()])
    target = torch.mean(norm_values)
    new_weights = {name: torch.clamp(target / torch.clamp(value, min=1e-12), 1e-3, 1e3).item() for name, value in norms.items()}
    for name, value in new_weights.items():
        loss_weights[name] = momentum * loss_weights[name] + (1.0 - momentum) * value


time1 = time.time()
print_every = max(1, epochs_joint // 20)

for epoch in range(1, epochs_joint + 1):
    t_data_batch, i_obs_batch, t_ode_batch = sample_batch(t_data_tensor, i_obs_tensor, t_ode_tensor, batch_size)
    losses = compute_losses(model, physics_informer, t_data_batch, i_obs_batch, t_ode_batch)

    if adaptive_NTK and epoch % adaptive_NTK["freq"] == 0:
        update_loss_weights(losses, model)

    total_loss = (
        loss_weights["data"] * losses["data"]
        + loss_weights["ode"] * losses["ode"]
        + loss_weights["ic"] * losses["ic"]
    )

    optimizer.zero_grad(set_to_none=True)
    total_loss.backward()
    optimizer.step()

    beta_value = model.beta().detach().cpu().item()
    history["loss"].append(total_loss.detach().cpu().item())
    history["loss_data"].append(losses["data"].detach().cpu().item())
    history["loss_ode"].append(losses["ode"].detach().cpu().item())
    history["loss_ic"].append(losses["ic"].detach().cpu().item())
    history["Beta"].append(beta_value)
    history["w_data"].append(loss_weights["data"])
    history["w_ode"].append(loss_weights["ode"])
    history["w_ic"].append(loss_weights["ic"])

    if epoch == 1 or epoch % print_every == 0 or epoch == epochs_joint:
        print(
            f"epoch {epoch:5d}/{epochs_joint} "
            f"loss={history['loss'][-1]:.3e} "
            f"data={history['loss_data'][-1]:.3e} "
            f"ode={history['loss_ode'][-1]:.3e} "
            f"ic={history['loss_ic'][-1]:.3e} "
            f"Beta={beta_value:.6f}"
        )

time2 = time.time()
print(f"Training time: {time2 - time1:.2f} seconds")

## Prediction and Visualization

The predictions are converted back to the original population scale by multiplying scaled states by `C`, exactly as in the original notebook.

In [ ]:
@torch.no_grad()
def predict_scaled(model, t_scaled):
    model.eval()
    ts = make_tensor(t_scaled)
    pred = model(ts)
    return {name: value.detach().cpu().numpy() for name, value in pred.items()}

pred_test = predict_scaled(model, t_test_sc)
S_pred_test = pred_test["Ss"] * C
I_pred_test = pred_test["Is"] * C
R_pred_test = pred_test["Rs"] * C

plt.plot(t_data, S_data, c="b", linewidth=4)
plt.plot(t_test, S_pred_test, "--", c="k", linewidth=4)
plt.xlabel("days")
plt.ylabel("individuals")
plt.legend([r"$S$", r"$\hat{S}$"])
plt.show()

plt.plot(t_data, I_data, c="r", linewidth=4)
plt.plot(t_test, I_pred_test, "--", c="k", linewidth=4)
if weekly:
    plt.scatter(t_data[::7], I_obs[::7], marker="x", c="m", s=100)
else:
    plt.scatter(t_data, I_obs, marker="x", c="m", s=100)
plt.xlabel("days")
plt.ylabel("individuals")
plt.legend([r"$I$", r"$\hat{I}$", "samples"])
plt.show()

plt.plot(t_data, R_data, c="g", linewidth=4)
plt.plot(t_test, R_pred_test, "--", c="k", linewidth=4)
plt.xlabel("days")
plt.ylabel("individuals")
plt.legend([r"$R$", r"$\hat{R}$"])
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history["Beta"], linewidth=2.5)
plt.axhline(beta, color="k", linestyle="--", linewidth=1.5, label="true beta")
plt.xlabel("epochs")
plt.ylabel(r"$\hat{\beta}_0$")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
pred_data = predict_scaled(model, t_data_sc)
S_pred = pred_data["Ss"] * C
I_pred = pred_data["Is"] * C
R_pred = pred_data["Rs"] * C
beta_pred = float(pred_data["Beta"][0, 0])

S_err = relative_l2(S_data, S_pred)
I_err = relative_l2(I_data, I_pred)
R_err = relative_l2(R_data, R_pred)
beta_err = abs(beta_pred - beta) / beta

print(f"S error: {S_err:.3e}")
print(f"I error: {I_err:.3e}")
print(f"R error: {R_err:.3e}")
print(f"Beta pred: {beta_pred:.6f}")
print(f"Beta error: {beta_err:.3e}")

## Save Training History and Model

The original notebook saves `hJoint.txt` and `mJoint.hdf5`. This PhysicsNeMo/PyTorch version saves a pickle training history and a `.pt` checkpoint with model and optimizer state.

In [ ]:
output_dir = Path(".")
history_path = output_dir / "hJoint-PhysicsNeMo-CUDA.pkl"
checkpoint_path = output_dir / "mJoint-PhysicsNeMo-CUDA.pt"

with history_path.open("wb") as handle:
    pickle.dump(history, handle)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "config": {
            "N": N,
            "delta": delta,
            "r0": r0,
            "beta_true": beta,
            "t0": t0,
            "tf": tf,
            "C": C,
            "C1": C1,
            "C2": C2,
            "Nc": Nc,
            "epochs_joint": epochs_joint,
            "batch_size": batch_size,
            "device": str(device),
        },
    },
    checkpoint_path,
)

print("Saved history to:", history_path.resolve())
print("Saved checkpoint to:", checkpoint_path.resolve())